# Finetuning QWEN-VL 7B for Speech Understanding

## 🌟 WHAT?

In this notebook, you will learn how to fine-tune [Qwen2-VL-7B](https://qwenlm.github.io/blog/qwen2-vl/) to understand speech using Hugging Face. Specifically, you will fine tune the model for the task of speech to text (a.k.a, automatic speech recognition).


💡 You can execute this Jupyter Notebook on a remote machine and then access and interact with it in your local web browser, leveraging the remote machine's computational resources.
- On remote: jupyter notebook --no-browser --port=8080
- On local: ssh -L 8080:localhost:8080 ntajbakhsh@workstation

🚨 **WARNING**: Please note that QWEN2-VL-7B is a relatively large model, requiring significant computational resources for fine-tuning. I recommend using either 2x A6000 or 1x A100 GPUs to ensure sufficient memory and processing power. While I haven't experimented with other GPUs, you're welcome to try alternative options. However, please be aware that other GPUs may not have enough memory to accommodate the model and optimizer states during training.

🚨 **WARNING**: Training transformers can be significantly more memory-efficient with Flash Attention (FA) compared to traditional attention mechanisms. However, FA support is currently limited to Nvidia's Ampere series of GPUs (A100, A6000, etc.) or better. If you're using an older GPU generation, please note that you'll need to disable FA to avoid error messages. Keep in mind that disabling FA may require using additional GPUs to compensate for the reduced memory efficiency.


# 1. Install Dependencies

Let’s start by installing the essential libraries we’ll need for fine-tuning! 🚀


In [1]:
!pip install -q -U transformers==4.47.0 git+https://github.com/huggingface/trl.git qwen-vl-utils datasets bitsandbytes peft  wandb accelerate matplotlib IPython librosa

ERROR: Cannot install transformers==4.47.0 and trl==0.23.0.dev0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


We will also need to install an earlier version of *PyTorch*, as the latest version has an issue that currently prevents this notebook from running correctly. You can learn more about the issue [here](https://github.com/pytorch/pytorch/issues/138340) and consider updating to the latest version once it’s resolved.

In [2]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

ERROR: Could not find a version that satisfies the requirement torch==2.4.1+cu121 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.4.1+cu121


As you will see later, you need to fork HF transformers and qwen-vl-utils to make necessary changes to handle audio. So, you need to reinstall HF transformers and qwen-vl-utils from your own forks. For the reinstall to take effect, your fork of transformer must be ahead of what was installed above.

In [ ]:
# Reinstall transformers and Qwen fork from your GitHub (replace branch if needed)
# This will upgrade/install directly from the specified forks so your local edits are picked up.
!python -m pip install -U pip setuptools wheel
!python -m pip install -U git+https://github.com/demirkeseny/transformers.git@main
!python -m pip install -U git+https://github.com/demirkeseny/Qwen2.5-VL.git@main

print("Done. Please restart the notebook kernel for changes to take effect.")

# 2. HF Login

Log in to Hugging Face to upload your fine-tuned model! 🗝️

You’ll need to authenticate with your Hugging Face account to save and share your model directly from this notebook.


In [ ]:
from huggingface_hub import login
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN') # add you token to colab secrets
import os
HF_TOKEN = os.environ['HF_TOKEN'] # export your HF_TOKEN first. You can add this to your ~/.bashrc.
login(token=HF_TOKEN)

## Optional Settings for an Improved Jupyter Experience

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%matplotlib inline

# 2. Understand Dataset 📁

Before we begin, take a moment to review this [link](https://huggingface.co/learn/audio-course/en/chapter1/introduction) and get familiar with the fundamentals of audio and speech data. This course covers the essentials of audio data, including the conversion of sound waves into digital formats, the role of sampling in analog-to-digital transformation, and the impact of sampling rates on audio quality. Additionally, you'll learn practical skills, such as loading and visualizing audio signals.

Next, you should load the [speechbrain/LargeScaleASR](https://huggingface.co/datasets/speechbrain/LargeScaleASR) dataset. This dataset contains 25,000 hours of diverse, transcribed English speech recognition data, available in three scalable sizes for research and commercial applications.

In [ ]:
# TASK: load the dataset in the streaming mode to avoid a full dataset download!


In [ ]:
# TASK: use take and grab 1 sample from the streaming dataset, check its content, and visual the wave using librosa



Now that we've covered the fundamentals, let's dive into using the OpenAI Whisper package to generate a sequence of audio tokens from a given audio signal. To do this effectively, make sure you understand the preprocessing step involved in generating the log mel spectrogram and how Whisper (the "turbo" variant) converts this intermediate representation into audio tokens.
Note that Whisper is an encoder-decoder model, but for our purposes, we'll only be using the encoder component to generate audio encodings.

In [ ]:
# TASK: Learn how to use whisper library to process an audio file from the dataset


# Pipeline we need to build

The diagram below shows the dataflow of training our audio-text model. We need to build all the required building blocks in this chapter.

![xxxx](https://www.dropbox.com/scl/fi/9dwf41lsz8xbvpw3kw6gh/VLA.drawio.png?rlkey=ono3hwexwog1511slkhsqk2g9&st=jyfao0xj&raw=1)


Before we begin, let's set up a Hugging Face repository. This will allow us to store and share the assets we create during this project.

In [ ]:
# TASK: Create a new repo on HF space and upload your new tokenizer


## Step #1: Format Dataset

We need to create a conversation (user/assistant) for each training sample. These conversations will then be consumed by model's apply chat template.

In [ ]:
#TASK: format the data for the model by brining each training sample into the OpenAI conversation format


In [ ]:
# TASK: apply the format_data function to the first sample of the dataset


## Step #2: Create chat template

The Qwen2-vl chat template is specifically designed to handle visual inputs. To expand its capabilities, we'll need to modify the template to also support audio inputs. Download Qwen2-vl's processor and assign your new template to processor.chat_template. Your template should use special tokens to indicate the start and end of audio input. So, you need to further extend processor's tokenizer by adding special tokens for handling audio:
- <|audio_start|>: Start of audio input
- <|audio_pad|>: Audio padding token
- <|audio_end|>: End of audio input

Once done, save the processor on a designated location on the disk, which saves the tokenizer as a folder. Push the folder to your HF space using api.upload_folder. Do not specify repo_type.

In [ ]:
# TASK: create a new chat template and save the processor


In [ ]:
# TASK: Push your new processor/tokenizer to your HF repo


In general, when increasing a tokenizer's vocabulary size, we need to resize the model's embedding and decoding layers. However, in this case, you don’t have to! Can you figure out why?

Now, let's load your processor from your HF repo and then apply the chat template and make sure it can properly handle a conversation with an audio input.

In [ ]:
# TASK: load Qwen2VLProcessor from your repo, then use it to process a formatted response


## Step #3: Load audio

To enable audio loading, you'll need to modify the vision_process.py file within the qwen-vl-utils package. Specifically, you should add a fetch_audio function, which will resemble the existing audio signal loading logic. This new function will be responsible for loading audio data from its byte-object representation into a NumPy array.

## Step #4: Extend the Processor

Qwen2-vl's processor tokenizes the chat template, repeats vision tokens for each image based on its size, and concatenates all vision inputs into a long tensor. To support audio, you'll need to modify this module to:
- Repeat the <|audio_pad|> token according to the length of each audio signal
- Concatenate all audio inputs into a long tensor
- Keep track of each audio input's length for later use in breaking the long input into individual audio encodings

To implement these changes, fork the transformers package and modify the processing_qwen2_vl.py file in src/transformers/models/qwen2_vl/.



Now, let's load your processor from your HF repo and also import process_vision_info to test the processor and fetch_audio functions.

In [ ]:
# TASK: load your processor and test how it works
from transformers import Qwen2VLProcessor
from qwen_vl_utils import process_vision_info


## Step #5: Create the Model

To create your model, follow the structure of Qwen2VLForConditionalGeneration. Specifically:
- Define a class that inherits from a PreTrained class, which simplifies model initialization, loading, and device mapping.
- Incorporate the audio encoder and audio projection layers into your model.

Implement your model class in src/transformers/models/qwen2_vl/modeling_qwen2_vl.py within your forked transformers repository.
Defining Model Configuration Classes

Additionally, define your model configuration classes in src/transformers/models/qwen2_vl/configuration_qwen2_vl.py. This will ensure your model is properly configured and aligned with the Qwen2VL architecture.

# Initialize & Push Model

Let's instantiate our model using the weights from the Qwen2VL checkpoint, noting that this will randomly initialize the audio encoder and audio projection layers. We'll then load the pre-trained weights from Whisper Turbo into our audio encoder, replacing the random initialization. After loading the Whisper Turbo weights, save the model, which now only has the audio projection layer randomly initialized. Finally, push the saved checkpoint to your Hugging Face repository. Now, both model and tokenizer should be there.

In [ ]:
# TASK: Load the Qwen2VL checkpoint into your model, load whisper turbo, save and push to your repo -  Now, the model and tokenizer should both be there.


One last step before we wrap the training pipeline is to update the model config, which has been created during saving the model in the last step. The current config may not include the special tokens for audio input. Also you may need to update _name_or_path in the config file.



In [ ]:
# TASK: Update and upload model_config to your HF repo


# Inference

Let's create a simple inference function that can take as the input a sample from the dataset and than
- convert to conversation format
- convert to chat template
- apply processing
- apply the model
- decode the generated tokens into output text


In [ ]:
# TASK: write an inference function for your model
import torch
import os
import torch
from qwen_vl_utils import vision_process
from qwen_vl_utils import process_vision_info


Test your inference script using this [image](https://t4.ftcdn.net/jpg/01/57/82/05/360_F_157820583_agejYX5XeczPZuWRSCDF2YYeCGwJqUdG.jpg) with the prompt: “Detect the bounding box of the red car.” The model should correctly identify and locate the car in the image, confirming the script’s correctness.

In [ ]:
# TASK: Run a vision-language test like what we had in the previous notebook with the VLA class to ensure original functionality is not broken
# You should get an identical output for a VL task with your model and Qwen2VL


Let's now see if our model, without any fine-tuning, can understand audio :)

In [ ]:
# Try your model with a conversation example containing audio


## Download Dataset

The dataset is quite large. Make sure to download it on a partition which has large enough space. For this, you can set export HF_DATASETS_CACHE="/your/custom/path" in your .bashrc file. Alternatively, you can train your model with a small shard of the dataset. I have done the latter, and here is the code I used. You may need to lower num_proc depending on how many CPU cores you have on your workstation.

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset("speechbrain/LargeScaleASR", data_files=["small/train-0000*","small/train-0001*"], num_proc=12)
test_dataset = load_dataset("speechbrain/LargeScaleASR", data_files=["test/test-00000*"], num_proc=12)
train_dataset = train_dataset["train"]
test_dataset = test_dataset["train"]
test_dataset = test_dataset.select(range(100)) # only 100 samples used for accelerated testing
print(len(train_dataset))
print(len(test_dataset))

## Clean up

Before we proceed with training the model in the next section, let's clear the current variables and clean the GPU to free up resources.

In [ ]:
import gc
import time
import torch
def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

# Model finetuning: Stage 1





Since the audio adapters are not trained yet, it makes sense to first freeze the language model and speech encoder and then train only the adapters.

In [ ]:
# TASK: Load model and processor from your HF repo, and freeze all params except audio projector. Don't use NF4 quantization for this task.


In [ ]:
from trl import SFTConfig
# TASK: create an SFT config



In [ ]:
import wandb
# TASK: set up wandb.init


In [ ]:
# TASK: Create a data collator to encode text and audio pairs


In [ ]:
# TASK: Create the SFT trainer and launch training



With the dataset above, I got the following train and eval loss.

![xxx](https://www.dropbox.com/scl/fi/ryjhfsrmn6topw8yqo8n4/phase1.png?rlkey=y80ovdst9nx3xuiniptmwv3vj&st=qfy05frr&raw=1)

In [ ]:
# TASk: push the finetuned model to your HF repo. Note how only 1 out of 4 sharded tensors has changed, why?


## Model finetuning: Stage 2

Now that we have trained the audio projection/adapter, we can finetune the entire model e2e with QLoRA.

QLoRA enables efficient fine-tuning of large language models while significantly reducing the memory footprint compared to traditional methods. Unlike standard LoRA, which reduces memory usage by applying a low-rank approximation, QLoRA takes it a step further by quantizing the model weights. This leads to even lower memory requirements and improved training efficiency, making it an excellent choice for optimizing our model's performance without sacrificing quality.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from trl import SFTTrainer
# Task: create LoRA, BitsAndBytes, and SFT configs and apply LoRA to the model



In [ ]:
# TASK: set up wand.init and create

In [ ]:
# TASK: Create the SFT trainer and launch training


With the dataset above, I got the following train and eval loss.

![xxx](https://www.dropbox.com/scl/fi/1whmbozm4pcog3ccw0y4g/phase2.png?rlkey=c0ui6y1ng48bjdgsp94p6wme2&st=e5ldl8lt&raw=1)

Let's save and push the results 💾

In [ ]:
# TASk: save and push the finetuned model to your HF repo


# 5. Testing the Fine-Tuned Model 🔍

Now that we've successfully fine-tuned our Audio-Language model, it's time to evaluate its performance! In this section, test the model using your own speech examples.

Recording and Preparing the Audio File
- Record an audio file: Use Voice Memos on Mac to record a speech example.
- Convert to WAV: Use  to convert the file to .wav.
- Resample the audio: Use torchaudio.transforms.Resample to resample the audio to 16K Hz, matching the model's training sampling rate.

Finally save it to an in-memory BytesIO object and include it in tour conversation template and feed to the model for transcription.


Let's clean up the GPU memory to ensure optimal performance 🧹

In [ ]:
clear_memory()

We will reload the base model using the same pipeline as before, but this we will load the LoRA adpaters into the model too.

In [ ]:
# TASK:  Load model, processor, and adapter weights


In [ ]:
import torchaudio
import torchaudio.transforms as T
import io
# TASK: use packages imported above to prepare your .wav file for ASR and run inference



Test the fine-tuned model on the example above, where the model previously struggled to accurately locate the nutrition table.